In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ouysse
from ouysse import *

In [ ]:
BASE       = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Alzou - Gramat\Gaetan"
PLUIE_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Pluie_BV_Ouysse.csv"

PRESTATAIRE   = os.path.join(BASE, "Alzou_consolide.xlsx")
PUNCTUAL      = os.path.join(BASE, "Punctual measurements .xlsx")
SORTIE_FINALE = os.path.join(BASE, "Alzou_final.xlsx")
SORTIE_SVG    = os.path.join(BASE, "Graphes.svg")

PAS = "1h"

#: {grandeur: colonne de l'export prestataire}
COLONNES = {
    "Niveau_(cm)":        "Niveau_(cm)",
    "Débit_(L/s)":        "Q_(L/s)",
    "Conductivité":       "Cond_Troll_(µS/cm)",
    "Température":        "température_(°C)",
    "Turbidité_(NTU)":    "Turbidity_Troll_(NTU)",
    "O2_(mg/l)":          "O2_Troll_(mg/l)",
    "Chlorophylle_(RFU)": "FluorescenceChloro_a_Troll_(RFU)",
}
PARAMETRES = list(COLONNES)

Station gérée par un prestataire : un seul export consolidé, pas de choix de sonde ni de fusion.

In [ ]:
brut = pd.read_excel(PRESTATAIRE).rename(columns={c: g for g, c in COLONNES.items()})
brut["DATE"] = pd.to_datetime(brut["DATE"], errors="coerce", dayfirst=True).dt.round(PAS)
for c in PARAMETRES:
    brut[c] = pd.to_numeric(brut[c].astype("string").str.strip()
                            .str.replace(",", ".", regex=False), errors="coerce").astype(float)
print(f"{brut['DATE'].notna().sum()} dates lues sur {len(brut)}, "
      f"jusqu'au {brut['DATE'].max():%d/%m/%Y %H:%M}")

full_data = sur_grille([empiler([brut], PARAMETRES)], PAS)

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data, {**GAMMES, "chloro": (0, 500), "o2": (0, 25)})

In [ ]:
#: (début, fin, colonne, motif) : mesures mises à l'écart.
VOIES_ECARTEES = [
    ("2023-09-05 11:00", "2023-10-24 11:00", "Conductivité", "sonde hors d'eau"),
]

print("Voies écartées :")
full_data = ecarter(full_data, VOIES_ECARTEES)

In [ ]:
points = lire_points(PUNCTUAL, col_jour="Date")

print("Points de contrôle :")
avant = full_data["Conductivité"]
full_data["Conductivité"] = caler(avant, points, "Conductivité")

graphe([(avant, "brute", "darkorange"),
        (full_data["Conductivité"], "calée", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)",
       points=points, col_point="Conductivité")

In [ ]:
FENETRE_IQR, K_IQR = "24h", 0.1   # k = 0 : pas de filtre
LISSAGE_H = 6                     # 0 = pas de lissage

avant = full_data["Conductivité"]
full_data["Conductivité"] = filtre_iqr(avant, FENETRE_IQR, K_IQR, lissage_h=LISSAGE_H)

graphe([(avant, "avant IQR et lissage", "darkorange"),
        (full_data["Conductivité"], "après IQR et lissage", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)")

In [ ]:
NIVEAU_NGF = None       # cote du zéro de l'échelle, None si elle n'est pas connue
MAX_TROU_H = 12

full_data = interpoler_avec_statut(full_data, PARAMETRES, MAX_TROU_H, PAS)

if NIVEAU_NGF is not None:
    full_data["Niveau_(mNGF)"] = NIVEAU_NGF + full_data["Niveau_(cm)"] / 100
    full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
    print(f"Zéro de l'échelle à {NIVEAU_NGF:.4f} m NGF")
display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

In [ ]:
finaux = [c for c in PARAMETRES + ["Niveau_(mNGF)"] if c in full_data]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}") if c in full_data]
sortie = full_data[colonnes].copy()
sortie.attrs["ouysse"] = ouysse.__version__
sortie.to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(sortie)} pas x {len(colonnes)} colonnes "
      f"(ouysse-hydro {ouysse.__version__})")

graphe_synthese(full_data, "Débit_(L/s)", "Débit (L/s)", pluie=PLUIE_PATH,
                sortie=SORTIE_SVG, marge_jours=15)

In [ ]:
graphe_statuts(full_data, finaux)